In [0]:
CREATE OR REPLACE TABLE support.bugzilla.gold_team_weekly AS
WITH weeks AS (   -- every Monday in 2024-2025 (the span our download covers)
  SELECT explode(sequence(DATE'2024-01-01', DATE'2025-12-29', INTERVAL 1 WEEK)) AS week_start
),
grid AS (         -- every team x every week, so quiet weeks still show up as 0
  SELECT t.team, w.week_start
  FROM (SELECT DISTINCT team FROM support.bugzilla.silver_bugs_all) t CROSS JOIN weeks w
)
SELECT g.team, g.week_start,
  COUNT_IF(b.created_at >= g.week_start AND b.created_at < date_add(g.week_start, 7)) AS opened,
  COUNT_IF(b.resolved_at >= g.week_start AND b.resolved_at < date_add(g.week_start, 7)) AS closed,
  COUNT_IF(b.created_at < date_add(g.week_start, 7)
           AND (b.resolved_at IS NULL OR b.resolved_at >= date_add(g.week_start, 7))) AS open_at_week_end,
  percentile_approx(CASE WHEN b.resolved_at >= g.week_start AND b.resolved_at < date_add(g.week_start, 7)
                         THEN datediff(b.resolved_at, b.created_at) END, 0.5) AS median_days_to_close
FROM grid g
LEFT JOIN support.bugzilla.silver_bugs_all b ON b.team = g.team
GROUP BY g.team, g.week_start

In [0]:
select * from support.bugzilla.gold_team_weekly order by week_start

In [0]:
select team, sum(closed) 
from support.bugzilla.gold_team_weekly
where week_start between '2025-12-08' and '2025-12-29'
group by team
order by sum(closed) desc

In [0]:
select team, sum(closed) / sum(opened)
from support.bugzilla.gold_team_weekly
group by team